# MCP version of the applicant application

The mcp servers use Rapid api to get linkedin posts, find the relevant posts, send an email to the user for the selected posts, with a curated cover letter and curated c.v


In [1]:
# imports
import os
import requests

from applicant import Applicant
from agents import Agent, Runner, trace
from agents.mcp import MCPServerSse
from templates import evaluation_instructions, listing_instructions
from dotenv import load_dotenv
from pypdf import PdfReader
from pathlib import Path
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
# set up profile
name = "Denis Gathondu"
denis = Applicant.get(name)
if not denis.summary:
    BASE_DIR = Path.cwd()
    profile_pdf_path = os.path.abspath(
        os.path.join(BASE_DIR, "sandbox", "linkedin.pdf")
    )
    with open(profile_pdf_path, "rb") as pdf_file:
        reader = PdfReader(pdf_file)
        profile = "\n".join([page.extract_text() for page in reader.pages])
    denis.set_summary(profile)


In [ ]:
rapid_api_key = os.getenv("RAPID_API_KEY")
rapid_api_host = os.getenv("RAPID_API_HOST")

url = f"https://{rapid_api_host}/active-jb-7d"

querystring = {
    "limit": "10",
    "offset": "0",
    "advanced_title_filter": "'AI Engineer' | Software:*",
    "location_filter": '"Nairobi"',
    "description_type": "text",
}

headers = {
    "x-rapidapi-key": rapid_api_key,
    "x-rapidapi-host": rapid_api_host,
    "Content-Type": "application/json",
}

response = requests.get(url, headers=headers, params=querystring)


In [ ]:
listing_message = f"""
Given the following profile and linked in results. Find me the relevant job posts for {name}.

profile: {denis}
linkedin_results: {response.json()}
"""

params = {"url": "http://127.0.0.1:8000/sse"}

async with MCPServerSse(params, client_session_timeout_seconds=30) as listing_server:
    listing_agent = Agent(
        name="listing_agent",
        instructions=listing_instructions(name),
        model="gpt-4o-mini",
        mcp_servers=[listing_server],
    )
    with trace("Listing Agent"):
        result = await Runner.run(listing_agent, listing_message)
    display(Markdown(result.final_output))


In [3]:
# Evaluator agent
job_posts = denis.list_job_posts()
evaluation_message = f"""
Evaluate each of the following job posts for {name}.

applicant profile: {denis}
job_post_ids: {[jp.id for jp in job_posts.job_posts]}
"""

params = {"url": "http://127.0.0.1:8000/sse"}

async with MCPServerSse(params, client_session_timeout_seconds=30) as eval_server:
    evaluation_agent = Agent(
        name="evaluation_agent",
        instructions=evaluation_instructions(name),
        model="gpt-4o-mini",
        mcp_servers=[eval_server],
    )
    with trace("Evaluation Agent"):
        result = await Runner.run(evaluation_agent, evaluation_message)
    display(Markdown(result.final_output))

I have evaluated the job posts for Denis Gathondu based on the provided applicant profile. Here are the results of the evaluations:

1. **Job Post ID: 1 - Senior Software Engineer at ABC Tech**
   - **Is Acceptable:** True
   - **Feedback:** Strong match with required skills in Python, Django, and React, which are critical for the role. The job requires 5+ years experience, aligning well with Denis's extensive background. The emphasis on collaboration with cross-functional teams complements Denis's leadership experience. Overall, a great fit for the position in Nairobi at ABC Tech.

2. **Job Post ID: 2071207383 - IT Software Services Delivery Manager at BCS Technology International**
   - **Is Acceptable:** False
   - **Feedback:** The role's focus on software delivery management and client relationships does not align well with Denis's primarily technical background and recent experience. There is insufficient overlap in must-have skills, especially in software delivery management. Therefore, this position is not a suitable fit.

3. **Job Post ID: 2077276264 - Head of Software Engineering at Nameless Ventures**
   - **Is Acceptable:** False
   - **Feedback:** The role requires leadership in a fintech environment and familiarity with Golang and event-driven systems, which do not align with Denis's current experience or tech stack, which mainly includes Python and related technologies. This position is not a match for Denis.

4. **Job Post ID: 2078454748 - Software Engineer - Cloud Images at Canonical**
   - **Is Acceptable:** True
   - **Feedback:** Denis's experience in Python and cloud technologies aligns well with Canonical's requirements for automation and collaboration on cloud infrastructure. The role's focus on software development and teamwork is a good match for his skill set. This position is suitable for Denis.

### Summary
- **Total Evaluations Completed**: 4
- **Acceptable Positions**: 2
- **Not Acceptable Positions**: 2

If you need further details or assistance, feel free to ask!

In [ ]:
len(denis.list_evaluations().evaluations)

4